In [ ]:
#zero shot marking (marking scheme only)
#“Single-pass rubric grading”, and it provides a baseline for comparison, with one LLM call producing
#a mark and feedback directly from the rubric and the essay text.

import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from types import SimpleNamespace
from langchain_core.messages import HumanMessage
from transformers import pipeline
from docx import Document
import re, json

In [ ]:
# Paths
ESSAY_DIR = "/path/to/project/essays"
MODEL_DIR = "/path/to/project/models/qwen2.5-7b"

# Prompt configuration
SYSTEM_PROMPT = """You are an academic tutor marking an essay.
Brief of the essay given to the students (follow strictly):
Write a reflective essay on your learning from this module.  The essay should be written in the first-person.
The essay will have two main areas of reflection:  
Reflect on learning,   
Reflect on application.  
In writing the essay you are aiming to strike a balance between your personal perspective, and the requirements of good academic practice and rigorous thinking.  
Common questions that such an essay will cover include:   
What were the most important ideas you learned, and why do you consider them important? 
What was your learning process, and what different methods helped with your learning? 
What went well and what went badly? 
What surprised you about what you learned? 
How can you apply what you learned to your work? 
Include in your reflection how you engaged with others to support your learning.  
For example: 
any discussion groups with fellow students,
discussions with work colleagues, members of your professional networks or university staff, 
and engaging in professional discussions on social media and/or the IPM Team channel.  
"""

MARKING_SCHEME = """Marking rubric (weights sum to 100):
- Overall presentation and quality of communication 20%, 
- Evidence of continuous reflection during your study 20% ,
- Breadth of study, including degree of engagement with others, including students and university staff 30%,
- Linking your studies with project management practices (both your own and more generally across the profession) 30%.
When marking consider the following ranges:
0–49 = Fail (criteria not met or met at an inadequate level)
50–59 = Pass (criteria met at a basic level)
60–69 = Merit (criteria met well, with good understanding and analysis)
70+ = Distinction (criteria met at an excellent level, showing originality, critical insight, and clear argument)
"""

JSON_RULE = (
    'Return exactly ONE valid JSON object like:\n'
    '{"score": 34, "feedback": "A short, specific critique."}\n'
    'Rules: "score" is an integer 0–100; "feedback" is one string; '
    'no extra text, no code fences, no reasoning.'
)

# Function to read docx files
def read_docx(path):
    doc = Document(path)
    return "\n".join(para.text for para in doc.paragraphs)

In [ ]:
# Load the model (4-bit NF4 quantisation)
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

#These parameters tell Hugging Face’s Transformers how to load the weights:
# load_in_4bit=True -> enables 4-bit quantization.
# bnb_4bit_compute_dtype=torch.float16 ->sets the compute precision during inference (float16 for speed).
# bnb_4bit_quant_type="nf4" -> specifies the “Normal Float 4” NF4” quant scheme to compress weights preserving more accuracy. 
# bnb_4bit_use_double_quant=True applies nested quantization squeezing memory further. 

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True, trust_remote_code=True)
#When running a model locallyy ou must load the exact tokenizer the model was trained with

# avoid pad warnings
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    quantization_config=bnb,
    device_map="auto",
    trust_remote_code=True,
)
#AutoModelForCausalLM is Hugging Face’s generic loader for any llm

# text-generation pipeline (handles device moves for you)
text_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    torch_dtype=torch.float16,
)

#Creates a Hugging Face pipeline object for text generation
#"text-generation": you want a generation pipeline (i.e., take a text prompt and produce a continuation). 
#model=model: passes in your already-loaded AutoModelForCausalLM.
#tokenizer=tokenizer: passes in your already-loaded tokenizer 
#device_map="auto" places the model’s layers on the GPU if possible, CPU otherwise)


In [ ]:
#Check if a template exists
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

print("Has chat_template?", bool(getattr(tokenizer, "chat_template", None)))
print((tokenizer.chat_template or "")[:300])  # quick peek



In [ ]:
def _invoke(messages, max_new_tokens=1024):
    prompt_text = messages[0].content if messages else ""
    out = text_pipeline(
        prompt_text,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.4,
        top_p=0.95,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        return_full_text=False,   # only the model’s output
    )
    return SimpleNamespace(content=out[0]["generated_text"].strip())


# drop-in equivalent to the API client
llm = SimpleNamespace(invoke=_invoke)


In [ ]:
for name in sorted(os.listdir(ESSAY_DIR)):
    if name.lower().endswith(".docx"):
        essay_text = read_docx(os.path.join(ESSAY_DIR, name)).strip()

        msgs = [
            {"role": "system", "content": f"{SYSTEM_PROMPT}\n\n{MARKING_SCHEME}"},
            {"role": "user", "content": f"Essay:\n{essay_text}\n\n{JSON_RULE}"},
        ]


        full_prompt = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        )
        response = llm.invoke([HumanMessage(content=full_prompt)])

        print(f"\n--- BEGIN {name} ---")          
        print(response.content)
        print(f"\n--- END {name} ---")
